# 04 - NYU Depth V2: ImageBind-style RGB-Depth Alignment

## FORMULATION

**Problem statement:** "Given paired RGB images and depth maps from NYU Depth V2, learn a shared embedding space where matching image-depth pairs are close and non-matching pairs are far."

**Input:** RGB image `I`, depth map `D`.

**Output:** normalized embeddings `z_I`, `z_D`.

**Task:** image-depth contrastive learning, image-to-depth retrieval, depth-to-image retrieval, and optional image-level scene classification only if scene-level labels such as `sceneTypes` can be parsed. Pixel-level semantic labels are not used as scene classes.

**Metrics:** Retrieval Recall@1, Recall@5, Recall@10; Top-1 / Top-5 accuracy if labels exist; train/validation loss.

## METHOD

**ImageBind core idea:** ImageBind learns a joint embedding space by using image as the anchor modality. Here RGB images anchor depth maps using paired `(image, depth)` samples from NYU Depth V2.

**Contrastive learning:** a positive pair is the matching RGB-depth pair. Negatives are the other samples in the mini-batch.

**InfoNCE loss:**

`L(I,D) = -log exp(sim(z_i, z_d)/tau) / sum_j exp(sim(z_i, z_d_j)/tau)`

We use symmetric loss:

`L = 0.5 * [L(image->depth) + L(depth->image)]`

Temperature `tau` controls the sharpness of similarity scores. A projection head maps encoder features to the shared embedding dimension. Depth is treated as a 1-channel image because it is a dense spatial map of geometry.

```text
RGB image -> Image Encoder -> Projection -> z_image
Depth map -> Depth Encoder -> Projection -> z_depth
z_image and z_depth -> symmetric InfoNCE loss
```

## IMPACT / Why this dataset matters

NYU Depth V2 is useful because depth represents 3D geometry and indoor scene structure. This complements previous modalities such as audio, text, and thermal. It shows that image-anchor contrastive learning can bind visual RGB with geometric depth.

Applications: indoor scene understanding, robotics, AR/VR, multimodal retrieval, and sensor fusion.

## 1. Setup & Imports

This notebook is built to run on Kaggle GPU T4/P100. The model is trained from scratch and does not use pretrained weights.

If `datasets` is missing and Kaggle Internet is enabled, set `INSTALL_MISSING_PACKAGES = True`. If Internet is disabled, add the dataset to Kaggle Input and the notebook will scan `/kaggle/input`.

In [ ]:
INSTALL_MISSING_PACKAGES = False

if INSTALL_MISSING_PACKAGES:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets", "transformers", "accelerate", "h5py"])

import os, json, glob, random, math, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image
import h5py
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

try:
    from IPython.display import display, Image as DisplayImage
except Exception:
    display = None
    DisplayImage = None

try:
    from datasets import load_dataset
    HF_DATASETS_AVAILABLE = True
except Exception as e:
    print("HuggingFace datasets is not available:", repr(e))
    HF_DATASETS_AVAILABLE = False
    load_dataset = None

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

## 2. Configuration

Use `quick_debug=True` for a short smoke test. The default settings train on a capped subset for reasonable Kaggle runtime.

In [ ]:
CONFIG = {
    "epochs": 5,
    "quick_debug": False,
    "max_train_samples": 3000,
    "max_val_samples": 600,
    "batch_size": 32,
    "image_size": 224,
    "embedding_dim": 256,
    "temperature": 0.07,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "num_workers": 2,
    "seed": 42,
    "base_channels": 32,
    "grad_clip_norm": 1.0,
    "classification_weight": 0.2,
    "max_ablation_batches": 100,
    "ablation_val_samples": 256,
}

if CONFIG["quick_debug"]:
    CONFIG.update({
        "epochs": 1,
        "max_train_samples": 512,
        "max_val_samples": 128,
        "batch_size": 16,
        "max_ablation_batches": 20,
        "ablation_val_samples": 128,
    })

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./kaggle_working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", device)
print("Mixed precision:", use_amp)
print("Output directory:", OUTPUT_DIR.resolve())
print(json.dumps(CONFIG, indent=2))

## 3. Dataset Loading

The Kaggle dataset version used for this project stores NYU Depth V2 in one MATLAB HDF5 file:

Path: /kaggle/input/.../NYU_depth_v2_labeled/nyu_depth_v2_labeled.mat

This section first auto-finds that .mat file with os.walk, opens it with h5py.File(mat_path, "r"), prints all keys and shapes, and uses images plus depths for the main RGB-depth retrieval task. Pixel-level semantic labels are not treated as image-level scene labels. If sceneTypes or another image-level label field can be parsed safely, classification is enabled; otherwise it is skipped.

If the .mat file is not found, the notebook falls back to the previous HuggingFace/local-file logic.

In [ ]:
IMAGE_COLUMN_CANDIDATES = ["image", "rgb", "color", "rgb_image", "image_rgb", "photo", "img"]
DEPTH_COLUMN_CANDIDATES = ["depth", "depth_map", "depth_image", "depthmap", "raw_depth", "metric_depth"]
LABEL_COLUMN_CANDIDATES = ["label", "scene", "scene_label", "class", "category", "room", "room_type"]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
DEPTH_EXTS = IMAGE_EXTS | {".npy", ".npz"}

raw_splits, source_name, source_type = None, None, None
mat_path = None
dataset_size = None
image_level_labels_raw = None
pixel_labels_available = False

def find_nyu_depth_mat(root="/kaggle/input"):
    for dirpath, dirnames, filenames in os.walk(root):
        for filename in filenames:
            if filename == "nyu_depth_v2_labeled.mat":
                return Path(dirpath) / filename
    return None

def print_h5_keys_and_shapes(path):
    print("HDF5 / MATLAB file:", path)
    with h5py.File(path, "r") as f:
        def visitor(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"{name}: shape={obj.shape}, dtype={obj.dtype}")
            else:
                print(f"{name}: group")
        f.visititems(visitor)

def decode_matlab_char_array(arr):
    arr = np.asarray(arr)
    if arr.dtype.kind in {"u", "i"}:
        chars = [chr(int(x)) for x in arr.reshape(-1) if int(x) != 0]
        return "".join(chars).strip()
    if arr.dtype.kind in {"S", "U"}:
        return "".join([str(x.decode("utf-8") if isinstance(x, bytes) else x) for x in arr.reshape(-1)]).strip()
    return None

def try_parse_scene_types(path, n_samples):
    """Parse optional image-level scene types from MATLAB/HDF5 references. Return None if unsupported."""
    candidate_keys = ["sceneTypes", "scene_types", "scenes", "sceneNames", "scene_names"]
    try:
        with h5py.File(path, "r") as f:
            key = next((k for k in candidate_keys if k in f), None)
            if key is None:
                return None
            ds = f[key]
            raw = ds[()]
            labels = []
            if raw.dtype == h5py.ref_dtype or raw.dtype.kind == "O":
                for ref in raw.reshape(-1):
                    if not ref:
                        labels.append(None)
                    else:
                        labels.append(decode_matlab_char_array(f[ref][()]))
            else:
                decoded = decode_matlab_char_array(raw)
                if decoded is not None and len(decoded) > 0:
                    labels = [decoded] * n_samples
            labels = [x if x not in ["", None] else None for x in labels]
            if len(labels) == n_samples and len(set([x for x in labels if x is not None])) >= 2:
                print(f"Parsed image-level labels from {key}: {len(set(labels))} classes")
                return labels
            print(f"Found {key}, but it could not be converted to {n_samples} image-level labels. Classification skipped.")
            return None
    except Exception as e:
        print("Could not parse image-level scene labels safely. Classification skipped.")
        print("Scene label parse error:", repr(e))
        return None

mat_path = find_nyu_depth_mat("/kaggle/input")
if mat_path is not None:
    source_type = "nyu_mat_h5"
    source_name = str(mat_path)
    print_h5_keys_and_shapes(mat_path)
    with h5py.File(mat_path, "r") as f:
        if "images" not in f or "depths" not in f:
            raise KeyError("Expected datasets 'images' and 'depths' in nyu_depth_v2_labeled.mat")
        dataset_size = int(f["images"].shape[0])
        if int(f["depths"].shape[0]) != dataset_size:
            raise ValueError(f"images/depths sample count mismatch: {f['images'].shape} vs {f['depths'].shape}")
        pixel_labels_available = "labels" in f
        print("Using f['images'] as RGB images")
        print("Using f['depths'] as depth maps")
        print("Pixel-level semantic labels available:", pixel_labels_available)
    image_level_labels_raw = try_parse_scene_types(mat_path, dataset_size)
    raw_splits = {"all_indices": list(range(dataset_size))}
    print(f"Loaded NYU Depth V2 MATLAB/HDF5 dataset with {dataset_size} samples.")

if raw_splits is None:
    if HF_DATASETS_AVAILABLE:
        try:
            raw_splits = load_dataset("sayakpaul/nyu_depth_v2")
            source_name = "sayakpaul/nyu_depth_v2"
            source_type = "huggingface"
            print("Loaded HuggingFace dataset:", source_name)
        except Exception as e:
            print("Could not load HuggingFace dataset. Will try local Kaggle input.")
            print("HF error:", repr(e))

def is_depth_like(path):
    text = str(path).lower().replace("\\", "/")
    return any(k in text for k in ["depth", "depth_map", "depthmap", "disparity"])

def is_rgb_like(path):
    text = str(path).lower().replace("\\", "/")
    return any(k in text for k in ["rgb", "image", "images", "color", "photo"])

def infer_label_from_path(path):
    parts = [p.lower() for p in Path(path).parts]
    blocked = {"kaggle", "input", "nyu", "nyu_depth", "nyu-depth", "nyu_depth_v2", "rgb", "image", "images", "color", "depth", "depths", "train", "val", "valid", "validation", "test"}
    for p in reversed(parts[:-1]):
        clean = p.replace(" ", "_")
        if clean and clean not in blocked and not clean.startswith("."):
            return clean
    return None

def possible_rgb_paths(depth_path):
    p = Path(depth_path)
    candidates = []
    replacements = [("depth_map", "image"), ("depthmap", "image"), ("depth", "rgb"), ("depth", "image"), ("depth", "color"), ("disparity", "rgb")]
    s = str(p)
    for old, new in replacements:
        lower = s.lower()
        if old in lower:
            idx = lower.find(old)
            candidates.append(Path(s[:idx] + new + s[idx + len(old):]))
    stem = p.stem.lower()
    for old, new in replacements:
        if old in stem:
            new_stem = stem.replace(old, new)
            for ext in IMAGE_EXTS:
                candidates.append(p.with_name(new_stem + ext))
    for ext in IMAGE_EXTS:
        candidates.append(p.with_suffix(ext))
    return candidates

def scan_kaggle_input(root="/kaggle/input"):
    root = Path(root)
    if not root.exists():
        print("Local input root does not exist:", root)
        return []
    all_files = [Path(x) for x in glob.glob(str(root / "**" / "*"), recursive=True) if Path(x).is_file()]
    depth_files = [p for p in all_files if p.suffix.lower() in DEPTH_EXTS and is_depth_like(p)]
    image_files = [p for p in all_files if p.suffix.lower() in IMAGE_EXTS and not is_depth_like(p)]
    image_lookup = {str(p).lower(): p for p in image_files}
    by_stem = defaultdict(list)
    for p in image_files:
        by_stem[p.stem.lower()].append(p)
    records = []
    for d in sorted(depth_files):
        matched = None
        for c in possible_rgb_paths(d):
            if str(c).lower() in image_lookup:
                matched = image_lookup[str(c).lower()]
                break
        if matched is None:
            stem = d.stem.lower()
            for token in ["depth_map", "depthmap", "depth", "disparity"]:
                stem = stem.replace(token, "")
            stem = stem.strip("_- .")
            if by_stem.get(stem):
                matched = by_stem[stem][0]
        if matched is not None:
            records.append({"image": str(matched), "depth": str(d), "label": infer_label_from_path(matched)})
    if len(records) == 0 and image_files and depth_files:
        print("No exact local pairs found; pairing sorted RGB-like files with sorted depth-like files by index.")
        rgb_candidates = [p for p in image_files if is_rgb_like(p)] or image_files
        for img, dep in zip(sorted(rgb_candidates), sorted(depth_files)):
            records.append({"image": str(img), "depth": str(dep), "label": infer_label_from_path(img)})
    return records

if raw_splits is None:
    local_records = scan_kaggle_input("/kaggle/input")
    if len(local_records) == 0:
        raise RuntimeError("Could not load NYU Depth V2. Add nyu_depth_v2_labeled.mat under /kaggle/input, enable HuggingFace Internet, or add RGB-depth files.")
    raw_splits = {"all": local_records}
    source_name = "/kaggle/input local scan"
    source_type = "local_files"
    print(f"Loaded {len(local_records)} local RGB-depth records.")

print("Source type:", source_type)
print("Source name:", source_name)
print("Split names:", list(raw_splits.keys()))

## 4. Dataset Inspection

For the MATLAB/HDF5 Kaggle file, this section confirms the detected sample count, the images and depths datasets, optional pixel-level semantic labels, and optional parsed image-level scene labels. For fallback sources, it prints split names, columns, and one example.

In [ ]:
if source_type == "nyu_mat_h5":
    print("Split names:", list(raw_splits.keys()))
    print("Total samples:", dataset_size)
    print("image_col: images")
    print("depth_col: depths")
    print("labels dataset is pixel-level semantic labels:", pixel_labels_available)
    print("Image-level scene labels parsed:", image_level_labels_raw is not None)
    with h5py.File(mat_path, "r") as f:
        print("images shape:", f["images"].shape, "dtype:", f["images"].dtype)
        print("depths shape:", f["depths"].shape, "dtype:", f["depths"].dtype)
        sample_img = f["images"][0]
        sample_dep = f["depths"][0]
        print("Sample image raw shape:", sample_img.shape, "dtype:", sample_img.dtype)
        print("Sample depth raw shape:", sample_dep.shape, "dtype:", sample_dep.dtype)
        if "labels" in f:
            print("labels shape:", f["labels"].shape, "dtype:", f["labels"].dtype)
    image_col = "images"
    depth_col = "depths"
    label_col = "sceneTypes" if image_level_labels_raw is not None else None
else:
    def get_columns(split_data):
        if source_type == "huggingface":
            return list(split_data.column_names)
        return list(split_data[0].keys()) if len(split_data) else []
    
    def detect_column(columns, candidates):
        lower_map = {c.lower(): c for c in columns}
        for cand in candidates:
            if cand.lower() in lower_map:
                return lower_map[cand.lower()]
        for c in columns:
            if any(cand.lower() in c.lower() for cand in candidates):
                return c
        return None
    
    split_names = list(raw_splits.keys())
    print("Split names:", split_names)
    for s in split_names:
        print(f"\nSplit: {s}")
        print("Rows:", len(raw_splits[s]))
        print("Columns:", get_columns(raw_splits[s]))
    
    first_split = raw_splits[split_names[0]]
    columns = get_columns(first_split)
    image_col = detect_column(columns, IMAGE_COLUMN_CANDIDATES) or ("image" if source_type == "local_files" else None)
    depth_col = detect_column(columns, DEPTH_COLUMN_CANDIDATES) or ("depth" if source_type == "local_files" else None)
    label_col = detect_column(columns, LABEL_COLUMN_CANDIDATES)
    
    print("\nDetected columns:")
    print("image_col:", image_col)
    print("depth_col:", depth_col)
    print("label_col:", label_col)
    if image_col is None or depth_col is None:
        raise ValueError(f"Could not detect required image/depth columns from {columns}")
    
    example = first_split[0]
    print("\nSample example keys:", list(example.keys()))
    for k, v in example.items():
        if k in [image_col, depth_col]:
            print(f"{k}: type={type(v)}, value_summary={v if isinstance(v, str) else 'object'}")
        elif k == label_col:
            print(f"{k}: {v}")

## 5. Preprocessing & Dataset Class

For nyu_depth_v2_labeled.mat, NYUDepthMatDataset opens the HDF5 file lazily inside the Dataset/worker instead of loading everything into RAM. It reads f["images"][idx] and f["depths"][idx], converts RGB to HWC/PIL, normalizes depth per image using 1st-99th percentiles, resizes both to 224x224, and returns image, depth, label, and index.

The .mat split is created by indices: 70% train, 15% validation, 15% test. Pixel-level labels are ignored for classification; image-level classification is only enabled if sceneTypes or another image-level field is parsed successfully.

In [ ]:
if source_type == "nyu_mat_h5":
    all_indices = np.arange(dataset_size)
    stratify = None
    if image_level_labels_raw is not None:
        vc = pd.Series(image_level_labels_raw).value_counts()
        if len(vc) > 1 and vc.min() >= 2:
            stratify = image_level_labels_raw

    train_indices, temp_indices = train_test_split(
        all_indices,
        train_size=0.70,
        random_state=CONFIG["seed"],
        shuffle=True,
        stratify=stratify,
    )
    temp_labels = [image_level_labels_raw[int(i)] for i in temp_indices] if image_level_labels_raw is not None else None
    temp_stratify = None
    if temp_labels is not None:
        vc_temp = pd.Series(temp_labels).value_counts()
        if len(vc_temp) > 1 and vc_temp.min() >= 2:
            temp_stratify = temp_labels
    val_indices, test_indices = train_test_split(
        temp_indices,
        test_size=0.50,
        random_state=CONFIG["seed"],
        shuffle=True,
        stratify=temp_stratify,
    )

    def limit_indices(indices, max_samples, seed):
        indices = np.asarray(indices)
        if max_samples is None or len(indices) <= max_samples:
            return indices.tolist()
        rng = np.random.default_rng(seed)
        return rng.choice(indices, size=max_samples, replace=False).astype(int).tolist()

    train_indices = limit_indices(train_indices, CONFIG["max_train_samples"], CONFIG["seed"])
    val_indices = limit_indices(val_indices, CONFIG["max_val_samples"], CONFIG["seed"] + 1)
    test_indices = limit_indices(test_indices, CONFIG["max_val_samples"], CONFIG["seed"] + 2)
    print("Prepared index splits:", {"train": len(train_indices), "val": len(val_indices), "test": len(test_indices)})

    if image_level_labels_raw is not None:
        unique = sorted(set([str(x) for x in image_level_labels_raw if x is not None]))
        label_to_idx = {v: i for i, v in enumerate(unique)}
        idx_to_label = {i: v for v, i in label_to_idx.items()}
    else:
        label_to_idx, idx_to_label = None, None

    has_labels = label_to_idx is not None and len(label_to_idx) >= 2
    num_classes = len(label_to_idx) if has_labels else 0
    if has_labels:
        print(f"Image-level classification enabled from parsed scene labels: {num_classes} classes")
    else:
        print("No image-level scene labels detected. Main task is RGB-depth retrieval; classification skipped.")

    def image_array_to_pil(arr):
        arr = np.asarray(arr)
        arr = np.squeeze(arr)
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        elif arr.ndim == 3:
            if arr.shape[0] in (1, 3, 4):
                arr = np.transpose(arr, (1, 2, 0))
            elif arr.shape[-1] not in (1, 3, 4) and arr.shape[1] in (1, 3, 4):
                arr = np.transpose(arr, (0, 2, 1))
            if arr.shape[-1] == 1:
                arr = np.repeat(arr, 3, axis=-1)
            if arr.shape[-1] > 3:
                arr = arr[..., :3]
        else:
            raise ValueError(f"Unsupported image array shape: {arr.shape}")
        if arr.dtype != np.uint8:
            arr = arr.astype(np.float32)
            if arr.max() <= 1.5:
                arr = arr * 255.0
            arr = np.clip(arr, 0, 255).astype(np.uint8)
        return Image.fromarray(arr).convert("RGB")

    def depth_array_to_pil(arr):
        arr = np.asarray(arr).astype(np.float32)
        arr = np.squeeze(arr)
        if arr.ndim == 3:
            arr = arr[..., 0]
        arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
        valid = arr[np.isfinite(arr)]
        if valid.size == 0:
            norm = np.zeros_like(arr, dtype=np.float32)
        else:
            lo, hi = np.percentile(valid, [1, 99])
            if hi <= lo:
                lo, hi = float(valid.min()), float(valid.max())
            norm = np.zeros_like(arr, dtype=np.float32) if hi <= lo else np.clip((arr - lo) / (hi - lo), 0, 1).astype(np.float32)
        return Image.fromarray((norm * 255).astype(np.uint8), mode="L")

    class NYUDepthMatDataset(Dataset):
        def __init__(self, mat_path, indices, image_level_labels=None, label_to_idx=None, train=False, image_size=224):
            self.mat_path = str(mat_path)
            self.indices = [int(i) for i in indices]
            self.image_level_labels = image_level_labels
            self.label_to_idx = label_to_idx
            self.train = train
            self.image_size = image_size
            self._h5 = None
            self.color_jitter = T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02)
            self.rgb_mean = torch.tensor([0.5, 0.5, 0.5]).view(3, 1, 1)
            self.rgb_std = torch.tensor([0.5, 0.5, 0.5]).view(3, 1, 1)

        def __len__(self):
            return len(self.indices)

        def __getstate__(self):
            state = self.__dict__.copy()
            state["_h5"] = None
            return state

        @property
        def h5(self):
            if self._h5 is None:
                self._h5 = h5py.File(self.mat_path, "r")
            return self._h5

        def __getitem__(self, idx):
            real_idx = int(self.indices[int(idx)])
            image = self.h5["images"][real_idx]
            depth = self.h5["depths"][real_idx]
            rgb = image_array_to_pil(image)
            depth = depth_array_to_pil(depth)

            if self.train:
                i, j, h, w = T.RandomResizedCrop.get_params(rgb, scale=(0.75, 1.0), ratio=(0.9, 1.1))
                rgb = TF.resized_crop(rgb, i, j, h, w, [self.image_size, self.image_size], interpolation=InterpolationMode.BILINEAR)
                depth = TF.resized_crop(depth, i, j, h, w, [self.image_size, self.image_size], interpolation=InterpolationMode.BILINEAR)
                if random.random() < 0.5:
                    rgb, depth = TF.hflip(rgb), TF.hflip(depth)
                rgb = self.color_jitter(rgb)
            else:
                rgb = TF.resize(rgb, [self.image_size, self.image_size], interpolation=InterpolationMode.BILINEAR)
                depth = TF.resize(depth, [self.image_size, self.image_size], interpolation=InterpolationMode.BILINEAR)

            image_tensor = (TF.to_tensor(rgb) - self.rgb_mean) / self.rgb_std
            depth_tensor = TF.to_tensor(depth).float()
            label = -1
            if self.image_level_labels is not None and self.label_to_idx is not None:
                raw_label = self.image_level_labels[real_idx]
                if raw_label is not None:
                    label = self.label_to_idx.get(str(raw_label), -1)
            return {
                "image": image_tensor,
                "depth": depth_tensor,
                "label": torch.tensor(label, dtype=torch.long),
                "index": torch.tensor(real_idx, dtype=torch.long),
            }

    train_dataset = NYUDepthMatDataset(mat_path, train_indices, image_level_labels_raw, label_to_idx, train=True, image_size=CONFIG["image_size"])
    val_dataset = NYUDepthMatDataset(mat_path, val_indices, image_level_labels_raw, label_to_idx, train=False, image_size=CONFIG["image_size"])
    test_dataset = NYUDepthMatDataset(mat_path, test_indices, image_level_labels_raw, label_to_idx, train=False, image_size=CONFIG["image_size"])

    train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

    batch = next(iter(train_loader))
    print("Batch image:", tuple(batch["image"].shape))
    print("Batch depth:", tuple(batch["depth"].shape))
    print("Batch index:", tuple(batch["index"].shape))
    print("Labels enabled:", has_labels)
else:
    def choose_train_val_test_splits(raw_splits, seed=42):
        names = list(raw_splits.keys())
        lower = {n.lower(): n for n in names}
        if source_type == "huggingface":
            if "train" in lower:
                train = raw_splits[lower["train"]]
                val_name = lower.get("validation") or lower.get("valid") or lower.get("val")
                test_name = lower.get("test")
                if val_name and test_name:
                    return train, raw_splits[val_name], raw_splits[test_name]
                if val_name:
                    return train, raw_splits[val_name], raw_splits[val_name]
                if test_name:
                    return train, raw_splits[test_name], raw_splits[test_name]
                split = train.train_test_split(test_size=0.2, seed=seed)
                vt = split["test"].train_test_split(test_size=0.5, seed=seed)
                return split["train"], vt["train"], vt["test"]
            first = raw_splits[names[0]]
            split = first.train_test_split(test_size=0.2, seed=seed)
            vt = split["test"].train_test_split(test_size=0.5, seed=seed)
            return split["train"], vt["train"], vt["test"]
    
        records = list(raw_splits[names[0]])
        labels = [r.get("label") for r in records]
        vc = pd.Series(labels).value_counts()
        stratify = labels if len(vc) > 1 and vc.min() >= 2 else None
        train_records, temp_records = train_test_split(records, test_size=0.2, random_state=seed, shuffle=True, stratify=stratify)
        temp_labels = [r.get("label") for r in temp_records]
        vc2 = pd.Series(temp_labels).value_counts()
        stratify2 = temp_labels if len(vc2) > 1 and vc2.min() >= 2 else None
        val_records, test_records = train_test_split(temp_records, test_size=0.5, random_state=seed, shuffle=True, stratify=stratify2)
        return train_records, val_records, test_records
    
    def limit_split(data, max_samples, seed=42):
        if max_samples is None or len(data) <= max_samples:
            return data
        if source_type == "huggingface":
            return data.shuffle(seed=seed).select(range(max_samples))
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(data), size=max_samples, replace=False)
        return [data[int(i)] for i in idx]
    
    train_raw, val_raw, test_raw = choose_train_val_test_splits(raw_splits, CONFIG["seed"])
    train_raw = limit_split(train_raw, CONFIG["max_train_samples"], CONFIG["seed"])
    val_raw = limit_split(val_raw, CONFIG["max_val_samples"], CONFIG["seed"] + 1)
    test_raw = limit_split(test_raw, CONFIG["max_val_samples"], CONFIG["seed"] + 2)
    print("Prepared splits:", {"train": len(train_raw), "val": len(val_raw), "test": len(test_raw)})
    
    def load_rgb(value):
        if isinstance(value, Image.Image):
            return value.convert("RGB")
        if isinstance(value, str):
            return Image.open(value).convert("RGB")
        arr = np.array(value)
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        if arr.dtype != np.uint8:
            arr = arr.astype(np.float32)
            arr = arr - np.nanmin(arr)
            arr = np.clip(arr / (np.nanmax(arr) + 1e-6) * 255.0, 0, 255).astype(np.uint8)
        return Image.fromarray(arr).convert("RGB")
    
    def load_depth_array(value):
        if isinstance(value, str):
            suffix = Path(value).suffix.lower()
            if suffix == ".npy":
                arr = np.load(value)
            elif suffix == ".npz":
                data = np.load(value)
                arr = data[data.files[0]]
            else:
                arr = np.array(Image.open(value))
        elif isinstance(value, Image.Image):
            arr = np.array(value)
        else:
            arr = np.array(value)
        arr = arr.astype(np.float32)
        if arr.ndim == 3:
            arr = arr[..., 0]
        return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    
    def normalize_depth_to_pil(depth_arr):
        valid = depth_arr[np.isfinite(depth_arr)]
        if valid.size == 0:
            norm = np.zeros_like(depth_arr, dtype=np.float32)
        else:
            lo, hi = np.percentile(valid, [1, 99])
            if hi <= lo:
                lo, hi = float(valid.min()), float(valid.max())
            norm = np.zeros_like(depth_arr, dtype=np.float32) if hi <= lo else np.clip((depth_arr - lo) / (hi - lo), 0, 1).astype(np.float32)
        return Image.fromarray((norm * 255).astype(np.uint8), mode="L")
    
    def get_label_value(example):
        if label_col is None or label_col not in example:
            return None
        v = example[label_col]
        if isinstance(v, (list, tuple, np.ndarray)):
            v = v[0] if len(v) else None
        return v
    
    def build_label_mapping(*splits):
        vals = []
        for split in splits:
            for i in range(len(split)):
                v = get_label_value(split[i])
                if v is not None:
                    vals.append(str(v))
        if not vals:
            return None, None
        unique = sorted(set(vals))
        label_to_idx = {v: i for i, v in enumerate(unique)}
        return label_to_idx, {i: v for v, i in label_to_idx.items()}
    
    label_to_idx, idx_to_label = build_label_mapping(train_raw, val_raw, test_raw) if label_col else (None, None)
    has_labels = label_to_idx is not None and len(label_to_idx) >= 2
    num_classes = len(label_to_idx) if has_labels else 0
    print(f"Detected labels: {num_classes} classes" if has_labels else "No usable labels found. Classification will be skipped.")
    
    class NYUDepthPairDataset(Dataset):
        def __init__(self, data, image_col, depth_col, label_to_idx=None, train=False, image_size=224):
            self.data = data
            self.image_col = image_col
            self.depth_col = depth_col
            self.label_to_idx = label_to_idx
            self.train = train
            self.image_size = image_size
            self.color_jitter = T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02)
            self.rgb_mean = torch.tensor([0.5, 0.5, 0.5]).view(3, 1, 1)
            self.rgb_std = torch.tensor([0.5, 0.5, 0.5]).view(3, 1, 1)
    
        def __len__(self):
            return len(self.data)
    
        def __getitem__(self, idx):
            ex = self.data[int(idx)]
            rgb = load_rgb(ex[self.image_col])
            depth = normalize_depth_to_pil(load_depth_array(ex[self.depth_col]))
            if self.train:
                i, j, h, w = T.RandomResizedCrop.get_params(rgb, scale=(0.75, 1.0), ratio=(0.9, 1.1))
                rgb = TF.resized_crop(rgb, i, j, h, w, [self.image_size, self.image_size], interpolation=InterpolationMode.BILINEAR)
                depth = TF.resized_crop(depth, i, j, h, w, [self.image_size, self.image_size], interpolation=InterpolationMode.BILINEAR)
                if random.random() < 0.5:
                    rgb, depth = TF.hflip(rgb), TF.hflip(depth)
                rgb = self.color_jitter(rgb)
            else:
                rgb = TF.resize(rgb, [self.image_size, self.image_size], interpolation=InterpolationMode.BILINEAR)
                depth = TF.resize(depth, [self.image_size, self.image_size], interpolation=InterpolationMode.BILINEAR)
            rgb_tensor = (TF.to_tensor(rgb) - self.rgb_mean) / self.rgb_std
            depth_tensor = TF.to_tensor(depth).float()
            label = -1
            if self.label_to_idx is not None:
                raw_label = get_label_value(ex)
                if raw_label is not None:
                    label = self.label_to_idx.get(str(raw_label), -1)
            return {"image": rgb_tensor, "depth": depth_tensor, "label": torch.tensor(label, dtype=torch.long), "index": torch.tensor(idx)}
    
    train_dataset = NYUDepthPairDataset(train_raw, image_col, depth_col, label_to_idx, train=True, image_size=CONFIG["image_size"])
    val_dataset = NYUDepthPairDataset(val_raw, image_col, depth_col, label_to_idx, train=False, image_size=CONFIG["image_size"])
    test_dataset = NYUDepthPairDataset(test_raw, image_col, depth_col, label_to_idx, train=False, image_size=CONFIG["image_size"])
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
    
    batch = next(iter(train_loader))
    print("Batch image:", tuple(batch["image"].shape))
    print("Batch depth:", tuple(batch["depth"].shape))
    print("Labels enabled:", has_labels)

## 6. Model Architecture

The model is a lightweight ImageBind-style dual encoder: a small ResNet-style RGB encoder, a matching 1-channel depth encoder, global average pooling, linear projection to 256 dimensions, L2 normalization, and an optional depth classifier.

In [ ]:
class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride=stride, padding=kernel_size // 2, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.net = nn.Sequential(ConvBNAct(channels, channels), nn.Conv2d(channels, channels, 3, padding=1, bias=False), nn.BatchNorm2d(channels))
        self.act = nn.SiLU(inplace=True)
    def forward(self, x):
        return self.act(x + self.net(x))

class SmallResNetEncoder(nn.Module):
    def __init__(self, in_channels=3, base_channels=32):
        super().__init__()
        c = base_channels
        self.net = nn.Sequential(
            ConvBNAct(in_channels, c, 7, 2), nn.MaxPool2d(3, stride=2, padding=1), ResidualBlock(c),
            ConvBNAct(c, c * 2, 3, 2), ResidualBlock(c * 2),
            ConvBNAct(c * 2, c * 4, 3, 2), ResidualBlock(c * 4),
            ConvBNAct(c * 4, c * 8, 3, 2), ResidualBlock(c * 8),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = c * 8
    def forward(self, x):
        return self.pool(self.net(x)).flatten(1)

class ProjectionHead(nn.Module):
    def __init__(self, in_dim, out_dim=256, kind="linear"):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, in_dim), nn.GELU(), nn.Linear(in_dim, out_dim)) if kind == "mlp" else nn.Linear(in_dim, out_dim)
    def forward(self, x):
        return self.net(x)

class MiniImageBindDepth(nn.Module):
    def __init__(self, embedding_dim=256, base_channels=32, num_classes=0, projection_kind="linear"):
        super().__init__()
        self.image_encoder = SmallResNetEncoder(3, base_channels)
        self.depth_encoder = SmallResNetEncoder(1, base_channels)
        enc_dim = self.image_encoder.out_dim
        self.image_proj = ProjectionHead(enc_dim, embedding_dim, projection_kind)
        self.depth_proj = ProjectionHead(enc_dim, embedding_dim, projection_kind)
        self.depth_classifier = nn.Linear(embedding_dim, num_classes) if num_classes and num_classes > 1 else None
    def forward(self, image, depth):
        z_img = F.normalize(self.image_proj(self.image_encoder(image)), dim=-1)
        z_dep = F.normalize(self.depth_proj(self.depth_encoder(depth)), dim=-1)
        logits = self.depth_classifier(z_dep) if self.depth_classifier is not None else None
        return z_img, z_dep, logits

model = MiniImageBindDepth(CONFIG["embedding_dim"], CONFIG["base_channels"], num_classes, "linear").to(device)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Train from scratch: true")
print("Pretrained weights: false")

## 7. InfoNCE / Symmetric Contrastive Loss

The batch forms an `N x N` similarity matrix. The diagonal contains positive pairs; off-diagonal entries are in-batch negatives.

In [ ]:
def symmetric_info_nce(z_img, z_depth, temperature=0.07):
    logits = (z_img @ z_depth.t()) / temperature
    targets = torch.arange(z_img.size(0), device=z_img.device)
    return 0.5 * (F.cross_entropy(logits, targets) + F.cross_entropy(logits.t(), targets))

def batch_loss(model, batch, temperature, classification_weight=0.2):
    image = batch["image"].to(device, non_blocking=True)
    depth = batch["depth"].to(device, non_blocking=True)
    labels = batch["label"].to(device, non_blocking=True)
    z_img, z_dep, logits = model(image, depth)
    contrastive = symmetric_info_nce(z_img, z_dep, temperature)
    ce = torch.tensor(0.0, device=device)
    if logits is not None and (labels >= 0).all():
        ce = F.cross_entropy(logits, labels)
    loss = contrastive + classification_weight * ce
    return loss, contrastive.detach(), ce.detach(), z_img, z_dep, logits

## 8. Training Loop

The loop uses mixed precision when CUDA is available, gradient clipping, validation loss, retrieval metrics, and checkpoint saving.

In [ ]:
def compute_retrieval_metrics(image_embeddings, depth_embeddings, ks=(1, 5, 10)):
    sim = np.asarray(image_embeddings, np.float32) @ np.asarray(depth_embeddings, np.float32).T
    n = sim.shape[0]
    metrics = {}
    for direction, scores in [("i2d", sim), ("d2i", sim.T)]:
        order = np.argsort(-scores, axis=1)
        for k in ks:
            kk = min(k, n)
            metrics[f"{direction}_r@{k}"] = float(np.mean([i in order[i, :kk] for i in range(n)]))
    return metrics

@torch.no_grad()
def extract_embeddings(model, loader):
    model.eval()
    img_embs, dep_embs, labels_all, logits_all = [], [], [], []
    for batch in tqdm(loader, desc="Extract embeddings", leave=False):
        image = batch["image"].to(device, non_blocking=True)
        depth = batch["depth"].to(device, non_blocking=True)
        z_img, z_dep, logits = model(image, depth)
        img_embs.append(z_img.cpu().numpy())
        dep_embs.append(z_dep.cpu().numpy())
        labels_all.append(batch["label"].numpy())
        if logits is not None:
            logits_all.append(logits.cpu().numpy())
    return (
        np.concatenate(img_embs, axis=0),
        np.concatenate(dep_embs, axis=0),
        np.concatenate(labels_all, axis=0),
        np.concatenate(logits_all, axis=0) if logits_all else None,
    )

@torch.no_grad()
def evaluate_loss(model, loader, temperature):
    model.eval()
    total = ctot = cetot = n = 0.0
    for batch in tqdm(loader, desc="Validation loss", leave=False):
        with torch.cuda.amp.autocast(enabled=use_amp):
            loss, contrastive, ce, *_ = batch_loss(model, batch, temperature, CONFIG["classification_weight"])
        bs = batch["image"].size(0)
        total += float(loss.item()) * bs
        ctot += float(contrastive.item()) * bs
        cetot += float(ce.item()) * bs
        n += bs
    return {"loss": total / max(n, 1), "contrastive_loss": ctot / max(n, 1), "ce_loss": cetot / max(n, 1)}

def classification_metrics(labels, logits, idx_to_label=None):
    if logits is None or labels is None:
        return {}
    mask = labels >= 0
    if mask.sum() == 0:
        return {}
    labels, logits = labels[mask], logits[mask]
    pred = logits.argmax(axis=1)
    top1 = float((pred == labels).mean())
    k = min(5, logits.shape[1])
    topk = np.argsort(-logits, axis=1)[:, :k]
    top5 = float(np.mean([labels[i] in topk[i] for i in range(len(labels))]))
    cm = confusion_matrix(labels, pred, labels=list(range(logits.shape[1])))
    per_class = {}
    for cls_idx in range(logits.shape[1]):
        denom = cm[cls_idx].sum()
        name = idx_to_label.get(cls_idx, str(cls_idx)) if idx_to_label else str(cls_idx)
        per_class[name] = float(cm[cls_idx, cls_idx] / denom) if denom > 0 else None
    return {"top1": top1, "top5": top5, "confusion_matrix": cm.tolist(), "per_class_accuracy": per_class}

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(CONFIG["epochs"], 1))
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
history = []
best_val_loss = float("inf")
checkpoint_path = OUTPUT_DIR / "nyu_depth_checkpoint.pth"

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    train_total = train_contrastive = train_ce = train_n = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{CONFIG['epochs']} train")
    for batch in pbar:
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            loss, contrastive, ce, *_ = batch_loss(model, batch, CONFIG["temperature"], CONFIG["classification_weight"])
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip_norm"])
        scaler.step(optimizer)
        scaler.update()
        bs = batch["image"].size(0)
        train_total += float(loss.item()) * bs
        train_contrastive += float(contrastive.item()) * bs
        train_ce += float(ce.item()) * bs
        train_n += bs
        pbar.set_postfix(loss=train_total / max(train_n, 1))

    scheduler.step()
    val_losses = evaluate_loss(model, val_loader, CONFIG["temperature"])
    val_img, val_dep, val_labels, val_logits = extract_embeddings(model, val_loader)
    retrieval = compute_retrieval_metrics(val_img, val_dep)
    cls_metrics = classification_metrics(val_labels, val_logits, idx_to_label) if has_labels else {}
    row = {
        "epoch": epoch,
        "train_loss": train_total / max(train_n, 1),
        "train_contrastive_loss": train_contrastive / max(train_n, 1),
        "train_ce_loss": train_ce / max(train_n, 1),
        "val_loss": val_losses["loss"],
        "val_contrastive_loss": val_losses["contrastive_loss"],
        "val_ce_loss": val_losses["ce_loss"],
        "lr": optimizer.param_groups[0]["lr"],
        **retrieval,
    }
    if cls_metrics:
        row["val_top1"] = cls_metrics["top1"]
        row["val_top5"] = cls_metrics["top5"]
    history.append(row)
    print(pd.DataFrame([row]).to_string(index=False))
    if row["val_loss"] < best_val_loss:
        best_val_loss = row["val_loss"]
        torch.save({"model_state_dict": model.state_dict(), "config": CONFIG, "epoch": epoch, "history": history, "label_to_idx": label_to_idx, "idx_to_label": idx_to_label}, checkpoint_path)
        print("Saved checkpoint:", checkpoint_path)

history_path = OUTPUT_DIR / "nyu_depth_train_history.json"
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)
print("Saved history:", history_path)

## 9. Evaluation: Retrieval and Classification

Evaluation is run on the test split. If the source only provided train/validation, validation is reused as test by the split logic.

In [ ]:
if checkpoint_path.exists():
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    print("Loaded best checkpoint from epoch", ckpt.get("epoch"))

val_img, val_dep, val_labels, val_logits = extract_embeddings(model, val_loader)
test_img, test_dep, test_labels, test_logits = extract_embeddings(model, test_loader)
val_retrieval = compute_retrieval_metrics(val_img, val_dep)
test_retrieval = compute_retrieval_metrics(test_img, test_dep)
test_cls = classification_metrics(test_labels, test_logits, idx_to_label) if has_labels else {}
final_loss = evaluate_loss(model, test_loader, CONFIG["temperature"])

eval_results = {
    "dataset": source_name,
    "source_type": source_type,
    "modality": "RGB-Depth",
    "train_from_scratch": True,
    "pretrained_weights": False,
    "epochs": CONFIG["epochs"],
    "temperature": CONFIG["temperature"],
    "test_loss": final_loss,
    "validation_retrieval": val_retrieval,
    "test_retrieval": test_retrieval,
    "classification": test_cls,
}
eval_path = OUTPUT_DIR / "nyu_depth_eval_results.json"
with open(eval_path, "w") as f:
    json.dump(eval_results, f, indent=2)

result_row = {
    "Dataset": source_name,
    "Modality": "RGB-Depth",
    "Epochs": CONFIG["epochs"],
    "Loss": final_loss["loss"],
    "I2D R@1": test_retrieval["i2d_r@1"],
    "I2D R@5": test_retrieval["i2d_r@5"],
    "I2D R@10": test_retrieval["i2d_r@10"],
    "D2I R@1": test_retrieval["d2i_r@1"],
    "D2I R@5": test_retrieval["d2i_r@5"],
    "D2I R@10": test_retrieval["d2i_r@10"],
    "Top1": test_cls.get("top1", None),
}
print(pd.DataFrame([result_row]).to_string(index=False))
print("Saved eval results:", eval_path)

## 10. Ablation: Temperature Sweep

This lightweight benchmark trains fresh models for a small number of mini-batches for temperatures `[0.05, 0.07, 0.1, 0.2]` and evaluates retrieval on a small validation subset.

In [ ]:
def make_subset_loader(dataset, max_samples, batch_size, shuffle, seed):
    n = min(len(dataset), max_samples)
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(dataset), size=n, replace=False).tolist() if len(dataset) > n else list(range(len(dataset)))
    return DataLoader(Subset(dataset, indices), batch_size=batch_size, shuffle=shuffle, num_workers=CONFIG["num_workers"], pin_memory=True, drop_last=shuffle)

@torch.no_grad()
def quick_retrieval_eval(eval_model, loader):
    img, dep, labels, logits = extract_embeddings(eval_model, loader)
    return compute_retrieval_metrics(img, dep)

def train_ablation_model(temperature, max_batches=100):
    seed_everything(CONFIG["seed"] + int(temperature * 1000))
    ab_model = MiniImageBindDepth(CONFIG["embedding_dim"], CONFIG["base_channels"], num_classes, "linear").to(device)
    opt = torch.optim.AdamW(ab_model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    sc = torch.cuda.amp.GradScaler(enabled=use_amp)
    losses = []
    ab_model.train()
    pbar = tqdm(enumerate(ablation_train_loader), total=min(max_batches, len(ablation_train_loader)), desc=f"Ablation tau={temperature}")
    for step, batch in pbar:
        if step >= max_batches:
            break
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            loss, *_ = batch_loss(ab_model, batch, temperature, CONFIG["classification_weight"])
        sc.scale(loss).backward()
        sc.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(ab_model.parameters(), CONFIG["grad_clip_norm"])
        sc.step(opt)
        sc.update()
        losses.append(float(loss.item()))
        pbar.set_postfix(loss=np.mean(losses))
    retrieval = quick_retrieval_eval(ab_model, ablation_val_loader)
    return {"temperature": temperature, "train_loss": float(np.mean(losses)) if losses else None, **retrieval}

ablation_train_loader = make_subset_loader(train_dataset, min(len(train_dataset), CONFIG["batch_size"] * CONFIG["max_ablation_batches"]), CONFIG["batch_size"], True, CONFIG["seed"] + 10)
ablation_val_loader = make_subset_loader(val_dataset, min(len(val_dataset), CONFIG["ablation_val_samples"]), CONFIG["batch_size"], False, CONFIG["seed"] + 11)

temperatures = [0.05, 0.07, 0.1, 0.2]
ablation_results = [train_ablation_model(tau, CONFIG["max_ablation_batches"]) for tau in temperatures]

ablation_path = OUTPUT_DIR / "nyu_depth_ablation_temperature.json"
with open(ablation_path, "w") as f:
    json.dump(ablation_results, f, indent=2)
print(pd.DataFrame(ablation_results).to_string(index=False))
print("Saved temperature ablation:", ablation_path)

## 11. Visualization

This section saves learning curves, retrieval metrics, optional confusion matrix, temperature ablation plot, and qualitative retrieval examples.

In [ ]:
def denormalize_rgb(t):
    t = torch.clamp(t.detach().cpu() * 0.5 + 0.5, 0, 1)
    return np.transpose(t.numpy(), (1, 2, 0))

def depth_to_show(t):
    return t.detach().cpu().squeeze(0).numpy()

learning_curve_path = OUTPUT_DIR / "nyu_depth_learning_curve.png"
if history:
    hdf = pd.DataFrame(history)
    plt.figure(figsize=(8, 5))
    plt.plot(hdf["epoch"], hdf["train_loss"], marker="o", label="train_loss")
    plt.plot(hdf["epoch"], hdf["val_loss"], marker="o", label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("NYU Depth RGB-Depth Training Curve")
    plt.legend()
    plt.tight_layout()
    plt.savefig(learning_curve_path, dpi=160)
    plt.show()
print("Saved:", learning_curve_path)

retrieval_metrics_path = OUTPUT_DIR / "nyu_depth_retrieval_metrics.png"
metric_names = ["R@1", "R@5", "R@10"]
i2d_vals = [test_retrieval["i2d_r@1"], test_retrieval["i2d_r@5"], test_retrieval["i2d_r@10"]]
d2i_vals = [test_retrieval["d2i_r@1"], test_retrieval["d2i_r@5"], test_retrieval["d2i_r@10"]]
x = np.arange(len(metric_names))
width = 0.35
plt.figure(figsize=(7, 5))
plt.bar(x - width / 2, i2d_vals, width, label="Image->Depth")
plt.bar(x + width / 2, d2i_vals, width, label="Depth->Image")
plt.xticks(x, metric_names)
plt.ylim(0, 1)
plt.ylabel("Recall")
plt.title("Retrieval Metrics")
plt.legend()
plt.tight_layout()
plt.savefig(retrieval_metrics_path, dpi=160)
plt.show()
print("Saved:", retrieval_metrics_path)

ablation_plot_path = OUTPUT_DIR / "nyu_depth_ablation_temperature.png"
adf = pd.DataFrame(ablation_results)
plt.figure(figsize=(8, 5))
plt.plot(adf["temperature"], adf["i2d_r@1"], marker="o", label="I2D R@1")
plt.plot(adf["temperature"], adf["d2i_r@1"], marker="o", label="D2I R@1")
plt.plot(adf["temperature"], adf["train_loss"], marker="s", label="Train loss")
plt.xlabel("Temperature tau")
plt.title("Temperature Ablation")
plt.legend()
plt.tight_layout()
plt.savefig(ablation_plot_path, dpi=160)
plt.show()
print("Saved:", ablation_plot_path)

confusion_matrix_path = OUTPUT_DIR / "nyu_depth_confusion_matrix.png"
if has_labels and test_cls.get("confusion_matrix") is not None:
    cm = np.array(test_cls["confusion_matrix"])
    plt.figure(figsize=(7, 6))
    plt.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.title("Depth Scene Classification Confusion Matrix")
    plt.colorbar()
    tick_count = min(num_classes, 20)
    ticks = np.arange(tick_count)
    tick_labels = [idx_to_label.get(i, str(i)) for i in ticks]
    plt.xticks(ticks, tick_labels, rotation=90, fontsize=7)
    plt.yticks(ticks, tick_labels, fontsize=7)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(confusion_matrix_path, dpi=160)
    plt.show()
    print("Saved:", confusion_matrix_path)
else:
    print("No labels available; confusion matrix skipped.")

retrieval_examples_path = OUTPUT_DIR / "nyu_depth_retrieval_examples.png"
def save_retrieval_examples(dataset, img_emb, dep_emb, path, num_queries=3, top_k=5):
    sim = img_emb @ dep_emb.T
    n = min(len(dataset), sim.shape[0])
    num_queries = min(num_queries, n)
    top_k = min(top_k, n)
    fig, axes = plt.subplots(num_queries * 2, top_k + 1, figsize=(2.1 * (top_k + 1), 3.8 * num_queries))
    axes = np.asarray(axes)
    for q in range(num_queries):
        sample = dataset[q]
        axes[2*q, 0].imshow(denormalize_rgb(sample["image"]))
        axes[2*q, 0].set_title(f"RGB query {q}")
        axes[2*q, 0].axis("off")
        for rank, idx in enumerate(np.argsort(-sim[q])[:top_k]):
            d_sample = dataset[int(idx)]
            axes[2*q, rank + 1].imshow(depth_to_show(d_sample["depth"]), cmap="magma")
            axes[2*q, rank + 1].set_title(f"D rank {rank+1}")
            axes[2*q, rank + 1].axis("off")
        axes[2*q + 1, 0].imshow(depth_to_show(sample["depth"]), cmap="magma")
        axes[2*q + 1, 0].set_title(f"Depth query {q}")
        axes[2*q + 1, 0].axis("off")
        for rank, idx in enumerate(np.argsort(-sim[:, q])[:top_k]):
            i_sample = dataset[int(idx)]
            axes[2*q + 1, rank + 1].imshow(denormalize_rgb(i_sample["image"]))
            axes[2*q + 1, rank + 1].set_title(f"I rank {rank+1}")
            axes[2*q + 1, rank + 1].axis("off")
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.show()

save_retrieval_examples(test_dataset, test_img, test_dep, retrieval_examples_path)
print("Saved:", retrieval_examples_path)

## 12. Save Outputs and Short Report

All required files are saved under `/kaggle/working` on Kaggle.

In [ ]:
best_temperature = max(ablation_results, key=lambda x: x.get("i2d_r@1", 0.0) + x.get("d2i_r@1", 0.0))["temperature"] if ablation_results else None

report_path = OUTPUT_DIR / "nyu_depth_report.txt"
report_lines = [
    "NYU Depth V2 RGB-Depth ImageBind-style Report",
    "===============================================",
    f"Dataset: {source_name}",
    "Modality: RGB-Depth",
    "Model: Mini ImageBind-style dual CNN encoders with projection heads",
    "Train from scratch: true",
    "Pretrained weights: false",
    f"Epochs: {CONFIG['epochs']}",
    f"Loss: {final_loss['loss']:.6f}",
    "Retrieval metrics:",
    f"  Image->Depth R@1:  {test_retrieval['i2d_r@1']:.4f}",
    f"  Image->Depth R@5:  {test_retrieval['i2d_r@5']:.4f}",
    f"  Image->Depth R@10: {test_retrieval['i2d_r@10']:.4f}",
    f"  Depth->Image R@1:  {test_retrieval['d2i_r@1']:.4f}",
    f"  Depth->Image R@5:  {test_retrieval['d2i_r@5']:.4f}",
    f"  Depth->Image R@10: {test_retrieval['d2i_r@10']:.4f}",
]
if has_labels and test_cls:
    report_lines += ["Classification metrics:", f"  Top-1: {test_cls.get('top1', 0.0):.4f}", f"  Top-5: {test_cls.get('top5', 0.0):.4f}"]
else:
    report_lines += ["Classification metrics: not available because labels were not found."]
report_lines += [
    f"Best temperature from ablation: {best_temperature}",
    "Short analysis:",
    "  The model learned image-depth alignment by pulling matched RGB-depth pairs together and pushing in-batch negatives apart.",
    "  Limitations: the model is intentionally small, trained on a capped subset, and does not use large-scale ImageBind pretraining.",
    "  Future improvement: train longer, add stronger synchronized augmentations, tune encoder capacity, and include more labeled scene supervision when labels are available.",
]
with open(report_path, "w") as f:
    f.write("\n".join(report_lines))

required_outputs = {
    "history": str(history_path),
    "eval_results": str(eval_path),
    "ablation_temperature": str(ablation_path),
    "learning_curve": str(learning_curve_path),
    "retrieval_metrics": str(retrieval_metrics_path),
    "retrieval_examples": str(retrieval_examples_path),
    "ablation_temperature_plot": str(ablation_plot_path),
    "report": str(report_path),
    "checkpoint": str(checkpoint_path),
}
if has_labels and confusion_matrix_path.exists():
    required_outputs["confusion_matrix"] = str(confusion_matrix_path)

print("Saved report:", report_path)
print("\n".join(report_lines))
print("\nOutput files:")
for name, path in required_outputs.items():
    print(f"{name}: {path}")

## 13. Final Summary

The final cell prints completion status, paths, final metrics, and displays generated plots.

In [ ]:
print("DONE: 04_nyu_depth_v2.ipynb completed")
print("\nSaved output files:")
for name, path in required_outputs.items():
    print(f"- {name}: {path}")

final_metrics = {
    "test_loss": final_loss,
    "test_retrieval": test_retrieval,
    "classification": {k: v for k, v in test_cls.items() if k != "confusion_matrix"} if test_cls else None,
    "best_temperature": best_temperature,
}
print("\nFinal metrics dictionary:")
print(json.dumps(final_metrics, indent=2))

if display is not None and DisplayImage is not None:
    for plot_path in [learning_curve_path, retrieval_metrics_path, retrieval_examples_path, ablation_plot_path, confusion_matrix_path]:
        if Path(plot_path).exists():
            print("Displaying:", plot_path)
            display(DisplayImage(filename=str(plot_path)))

## ADDON: Rubric completion and missing components

This addon preserves all original cells and only standardizes missing rubric deliverables at the end of the notebook.

It writes new files to /kaggle/working/imagebind_rubric_outputs/04_nyu_depth_v2/ on Kaggle, with a local ./kaggle_working fallback. Optional visualizations and ablations are wrapped in try/except; if the needed runtime variables are unavailable, the addon writes NOT_RUN or FAILED_WITH_REASON instead of inventing results.

NYU addon standardizes RGB-depth retrieval, optional classification outputs, temperature ablation, second-ablation status, retrieval example status, and rubric report.

In [ ]:
from pathlib import Path
import json, traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT_OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./kaggle_working")
ADDON_DIR = ROOT_OUT / "imagebind_rubric_outputs" / "04_nyu_depth_v2"
ADDON_DIR.mkdir(parents=True, exist_ok=True)
FAST_ABLATION = True

def _jsonable(x):
    if isinstance(x, dict): return {str(k): _jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)): return [_jsonable(v) for v in x]
    if hasattr(x, "tolist"): return x.tolist()
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.floating,)): return float(x)
    return x

def _load_json_candidates(names):
    roots = [ROOT_OUT, Path.cwd(), Path("ResultForDepth4")]
    for root in roots:
        for name in names:
            p = root / name
            if p.exists():
                with open(p, "r", encoding="utf-8") as f:
                    return json.load(f), str(p)
    return None, None

eval_json, eval_source = _load_json_candidates(["nyu_depth_eval_results.json"])
ab_json, ab_source = _load_json_candidates(["nyu_depth_ablation_temperature.json"])
hist_json, hist_source = _load_json_candidates(["nyu_depth_train_history.json"])

runtime_eval = globals().get("eval_results", None)
eval_obj = runtime_eval if isinstance(runtime_eval, dict) else (eval_json or {})
test_retrieval_obj = globals().get("test_retrieval", None)
retrieval = test_retrieval_obj if isinstance(test_retrieval_obj, dict) else eval_obj.get("test_retrieval", {})
classification = globals().get("test_cls", None) if isinstance(globals().get("test_cls", None), dict) else eval_obj.get("classification", {})

eval_summary = {
    "dataset": "NYU Depth V2",
    "modality": "RGB-Depth",
    "model_name": "MiniImageBindDepth",
    "train_from_scratch": True,
    "pretrained_weights": False,
    "epochs": globals().get("CONFIG", {}).get("epochs", eval_obj.get("epochs", "UNKNOWN")) if isinstance(globals().get("CONFIG", {}), dict) else eval_obj.get("epochs", "UNKNOWN"),
    "loss": "symmetric InfoNCE / contrastive loss with optional CE for image-level labels",
    "main_metrics": {k: float(v) for k, v in retrieval.items() if isinstance(v, (int, float))},
    "classification_metrics": classification if classification else "SKIPPED_IF_NO_IMAGE_LEVEL_LABELS",
    "ablation_status": "PRESENT_OR_STANDARDIZED",
    "limitation": "Mini CNN trained from scratch on a capped subset; no pretrained ImageBind/OpenCLIP, so low retrieval is expected.",
    "sources": {"eval": eval_source, "ablation": ab_source, "history": hist_source},
}
with open(ADDON_DIR / "final_eval_summary.json", "w", encoding="utf-8") as f:
    json.dump(_jsonable(eval_summary), f, indent=2)

ablation_summary = globals().get("ablation_results", None) if isinstance(globals().get("ablation_results", None), list) else (ab_json or [{"status": "NOT_RUN", "reason": "No temperature ablation found."}])
second_ablation = {"status": "NOT_RUN", "type": "batch_size_or_lr_smoke_test", "reason": "Skipped by default to avoid retraining NYU Depth V2. Temperature ablation is already present; set FAST_ABLATION=False and add a short training loop if more runtime is available."}
summary_payload = {"temperature_ablation": ablation_summary, "second_ablation": second_ablation}
with open(ADDON_DIR / "nyu_depth_ablation_summary.json", "w", encoding="utf-8") as f:
    json.dump(_jsonable(summary_payload), f, indent=2)
with open(ADDON_DIR / "ablation_summary.json", "w", encoding="utf-8") as f:
    json.dump(_jsonable(summary_payload), f, indent=2)

try:
    keys = ["i2d_r@1", "i2d_r@5", "i2d_r@10", "d2i_r@1", "d2i_r@5", "d2i_r@10"]
    vals = [retrieval.get(k, np.nan) for k in keys]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(["I2D R@1","I2D R@5","I2D R@10","D2I R@1","D2I R@5","D2I R@10"], vals)
    ax.set_ylim(0, 1); ax.set_ylabel("Recall"); ax.set_title("NYU RGB-Depth Retrieval")
    ax.tick_params(axis="x", rotation=25); fig.tight_layout()
    fig.savefig(ADDON_DIR / "nyu_depth_retrieval_bar.png", dpi=160)
    fig.savefig(ADDON_DIR / "retrieval_bar.png", dpi=160)
    plt.show()
except Exception as exc:
    print("Retrieval bar skipped:", exc)

try:
    df_ab = pd.DataFrame(ablation_summary)
    if "temperature" in df_ab.columns:
        fig, ax = plt.subplots(figsize=(7, 4))
        for col in ["i2d_r@1", "d2i_r@1"]:
            if col in df_ab: ax.plot(df_ab["temperature"], df_ab[col], marker="o", label=col)
        ax.set_title("NYU Temperature Ablation"); ax.set_xlabel("Temperature"); ax.set_ylabel("Recall@1"); ax.legend()
        fig.tight_layout(); fig.savefig(ADDON_DIR / "ablation_summary.png", dpi=160); plt.show()
except Exception as exc:
    print("Ablation plot skipped:", exc)

retrieval_examples_status = {"status": "PRESENT" if (ROOT_OUT / "nyu_depth_retrieval_examples.png").exists() or Path("ResultForDepth4/nyu_depth_retrieval_examples.png").exists() else "NOT_RUN", "note": "Original notebook saves qualitative RGB-depth retrieval examples when run end-to-end."}
with open(ADDON_DIR / "retrieval_examples.txt", "w", encoding="utf-8") as f:
    f.write(json.dumps(retrieval_examples_status, indent=2))

report = f"""NYU Depth V2 rubric addon report
================================
Overview: RGB-depth ImageBind-style alignment with RGB image as anchor and depth aligned into the shared space.
Method: RGB and depth encoders are trained from scratch with symmetric InfoNCE on paired samples. Depth is treated as a 1-channel geometric image.
Implementation: train_from_scratch=True, pretrained_weights=False, epochs={eval_summary['epochs']}.
Evaluation: {json.dumps(eval_summary['main_metrics'], indent=2)}
Classification: {json.dumps(classification, indent=2) if classification else 'Skipped unless image-level labels are detected.'}
Ablation: temperature ablation standardized to nyu_depth_ablation_summary.json; second ablation is recorded as NOT_RUN by default to keep Kaggle runtime reasonable.
Limitations: mini CNN, capped subset, no pretrained ImageBind/OpenCLIP; low retrieval is reasonable for this setting.
Future work: train longer, tune depth normalization, add stronger synchronized augmentation, and align with text/image anchors across datasets.
"""
with open(ADDON_DIR / "nyu_depth_rubric_report.txt", "w", encoding="utf-8") as f:
    f.write(report)
print("ADDON outputs:", ADDON_DIR)
print(json.dumps(eval_summary, indent=2))

## ADDON: Export standardized outputs

This cell standardizes this notebook's outputs into its required project folder. It copies available files, creates a short README/report if needed, and never crashes when optional outputs are missing.

In [ ]:
from pathlib import Path
import json
import shutil
import traceback

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./kaggle_working")
STANDARD_OUTPUT_DIR = WORKING_DIR / "ResultForDepth4"
STANDARD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_STEM = "04_nyu_depth_v2"
DATASET_NAME = "NYU Depth V2"
MODALITY_NAME = "RGB-Depth"
MAX_CHECKPOINT_BYTES = 100 * 1024 * 1024
CHECKPOINT_SUFFIXES = {".pth", ".pt", ".ckpt"}

FILE_EXPORTS = {
    "train_history.json": ["train_history.json", "nyu_depth_train_history.json"],
    "eval_results.json": ["eval_results.json", "nyu_depth_eval_results.json"],
    "final_eval_summary.json": ["final_eval_summary.json"],
    "retrieval_recall.json": ["retrieval_recall.json", "nyu_depth_eval_results.json"],
    "ablation_summary.json": ["ablation_summary.json", "nyu_depth_ablation_summary.json", "nyu_depth_ablation_temperature.json"],
    "learning_curve.png": ["learning_curve.png", "nyu_depth_learning_curve.png"],
    "confusion_matrix.png": ["confusion_matrix.png", "nyu_depth_confusion_matrix.png"],
    "report.txt": ["report.txt", "nyu_depth_report.txt", "nyu_depth_rubric_report.txt"],
    "rubric_report.txt": ["rubric_report.txt", "nyu_depth_rubric_report.txt"],
    "retrieval_examples.png": ["retrieval_examples.png", "nyu_depth_retrieval_examples.png"],
    "nyu_depth_retrieval_bar.png": ["nyu_depth_retrieval_bar.png", "retrieval_bar.png", "nyu_depth_retrieval_metrics.png"],
}

def _as_path(value):
    try:
        return Path(value)
    except Exception:
        return None

def _candidate_dirs():
    dirs = [
        WORKING_DIR,
        WORKING_DIR / "outputs",
        WORKING_DIR / "results",
        WORKING_DIR / "figures",
        WORKING_DIR / "imagebind_rubric_outputs" / NOTEBOOK_STEM,
        Path("."),
        Path("outputs"),
        Path("results"),
        Path("figures"),
        Path("ResultForDepth4"),
    ]
    for var_name in ["OUTPUT_DIR", "RESULT_DIR", "FIGURE_DIR"]:
        if var_name in globals():
            p = _as_path(globals()[var_name])
            if p is not None:
                dirs.append(p)
                dirs.append(p / "results")
                dirs.append(p / "figures")
    seen = set()
    out = []
    for d in dirs:
        try:
            key = str(d.resolve()) if d.exists() else str(d)
        except Exception:
            key = str(d)
        if key not in seen:
            out.append(d)
            seen.add(key)
    return out

def _skip_reason(path):
    try:
        if path.suffix.lower() in CHECKPOINT_SUFFIXES and path.stat().st_size > MAX_CHECKPOINT_BYTES:
            return f"large checkpoint >100MB ({path.stat().st_size / (1024**2):.1f} MB)"
    except Exception as exc:
        return f"stat failed: {exc}"
    return None

def _copy_file(src, dst):
    reason = _skip_reason(src)
    if reason:
        return {"source": str(src), "dest": str(dst), "status": "SKIPPED", "reason": reason}
    try:
        dst.parent.mkdir(parents=True, exist_ok=True)
        try:
            if src.resolve() == dst.resolve():
                return {"source": str(src), "dest": str(dst), "status": "ALREADY_IN_PLACE"}
        except Exception:
            pass
        shutil.copy2(src, dst)
        return {"source": str(src), "dest": str(dst), "status": "COPIED"}
    except Exception as exc:
        return {"source": str(src), "dest": str(dst), "status": "FAILED", "reason": repr(exc)}

def _find_source(names):
    for base in _candidate_dirs():
        for name in names:
            p = Path(name)
            candidates = []
            if p.is_absolute():
                candidates.append(p)
            else:
                candidates.append(base / name)
                try:
                    if base.exists():
                        candidates.extend(base.rglob(name))
                except Exception:
                    pass
            for cand in candidates:
                if cand.exists() and cand.is_file():
                    return cand
    return None

manifest = []
for dest_name, source_names in FILE_EXPORTS.items():
    src = _find_source(source_names)
    dst = STANDARD_OUTPUT_DIR / dest_name
    if src is None:
        msg = {"dest": dest_name, "status": "MISSING", "searched_names": source_names}
        manifest.append(msg)
        print("WARNING missing output for", dest_name, "searched:", source_names)
        continue
    manifest.append(_copy_file(src, dst))

# Copy any extra rubric addon files into the standardized folder without overwriting canonical names.
addon_dir = WORKING_DIR / "imagebind_rubric_outputs" / NOTEBOOK_STEM
if addon_dir.exists():
    for src in sorted(addon_dir.rglob("*")):
        if not src.is_file():
            continue
        dst = STANDARD_OUTPUT_DIR / src.name
        if dst.exists():
            continue
        manifest.append(_copy_file(src, dst))

if not (STANDARD_OUTPUT_DIR / "final_eval_summary.json").exists() and not (STANDARD_OUTPUT_DIR / "eval_results.json").exists():
    fallback_summary = {
        "dataset": DATASET_NAME,
        "modality": MODALITY_NAME,
        "status": "NOT_RUN",
        "reason": "No eval_results.json or final_eval_summary.json was found by the export addon.",
    }
    with open(STANDARD_OUTPUT_DIR / "final_eval_summary.json", "w", encoding="utf-8") as f:
        json.dump(fallback_summary, f, indent=2)
    manifest.append({"dest": "final_eval_summary.json", "status": "CREATED_NOT_RUN"})

if not any((STANDARD_OUTPUT_DIR / name).exists() for name in ["report.txt", "rubric_report.txt"]):
    report = f"""Standardized output report
Dataset: {DATASET_NAME}
Modality: {MODALITY_NAME}
Notebook: {NOTEBOOK_STEM}
Status: exported available files. Missing files are listed in export_manifest.json.
"""
    with open(STANDARD_OUTPUT_DIR / "report.txt", "w", encoding="utf-8") as f:
        f.write(report)
    manifest.append({"dest": "report.txt", "status": "CREATED_MINIMAL"})

readme = f"""# {NOTEBOOK_STEM} standardized outputs

Dataset: {DATASET_NAME}
Modality: {MODALITY_NAME}
Folder: {STANDARD_OUTPUT_DIR}

This folder is created by the export addon. Missing optional files are warnings, not fatal errors.
See export_manifest.json for copied, missing, and skipped files.
"""
with open(STANDARD_OUTPUT_DIR / "README_outputs.md", "w", encoding="utf-8") as f:
    f.write(readme)

with open(STANDARD_OUTPUT_DIR / "export_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Standardized output directory:", STANDARD_OUTPUT_DIR)
print("Exported files now present:")
for p in sorted(STANDARD_OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print("-", p.relative_to(STANDARD_OUTPUT_DIR))

## ADDON: Excel report

This cell creates a styled Excel workbook from JSON/TXT artifacts already exported into this notebook output folder. Missing files are recorded as NOT_AVAILABLE instead of stopping the notebook.

In [ ]:
from pathlib import Path
import json

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./kaggle_working")
EXCEL_OUTPUT_DIR = WORKING_DIR / "ResultForDepth4"
EXCEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXCEL_REPORT_PATH = EXCEL_OUTPUT_DIR / "04_nyu_depth_excel_report.xlsx"

METRIC_SET = "nyu"

METRIC_SPECS = {
    "esc50": [
        ("test_top1", ["test_top1", "top1", "top1_acc", "test_accuracy"], "Test top-1 classification accuracy"),
        ("test_top5", ["test_top5", "top5", "top5_acc"], "Test top-5 classification accuracy"),
        ("audio_to_text_R@1", ["audio_to_text_R@1", "audio_to_text_r1", "a2t_R@1"], "Audio-to-text Recall@1"),
        ("audio_to_text_R@5", ["audio_to_text_R@5", "audio_to_text_r5", "a2t_R@5"], "Audio-to-text Recall@5"),
        ("audio_to_text_R@10", ["audio_to_text_R@10", "audio_to_text_r10", "a2t_R@10"], "Audio-to-text Recall@10"),
        ("text_to_audio_R@1", ["text_to_audio_R@1", "text_to_audio_r1", "t2a_R@1"], "Text-to-audio Recall@1"),
        ("text_to_audio_R@5", ["text_to_audio_R@5", "text_to_audio_r5", "t2a_R@5"], "Text-to-audio Recall@5"),
        ("text_to_audio_R@10", ["text_to_audio_R@10", "text_to_audio_r10", "t2a_R@10"], "Text-to-audio Recall@10"),
    ],
    "flickr8k": [
        ("image_to_text_R@1", ["image_to_text_R@1", "image_to_text_r1", "i2t_R@1", "I2T R@1"], "Image-to-text Recall@1"),
        ("image_to_text_R@5", ["image_to_text_R@5", "image_to_text_r5", "i2t_R@5", "I2T R@5"], "Image-to-text Recall@5"),
        ("image_to_text_R@10", ["image_to_text_R@10", "image_to_text_r10", "i2t_R@10", "I2T R@10"], "Image-to-text Recall@10"),
        ("text_to_image_R@1", ["text_to_image_R@1", "text_to_image_r1", "t2i_R@1", "T2I R@1"], "Text-to-image Recall@1"),
        ("text_to_image_R@5", ["text_to_image_R@5", "text_to_image_r5", "t2i_R@5", "T2I R@5"], "Text-to-image Recall@5"),
        ("text_to_image_R@10", ["text_to_image_R@10", "text_to_image_r10", "t2i_R@10", "T2I R@10"], "Text-to-image Recall@10"),
    ],
    "llvip": [
        ("image_to_thermal_R@1", ["image_to_thermal_R@1", "image_to_thermal_r1", "i2t_R@1"], "Visible image-to-thermal Recall@1"),
        ("image_to_thermal_R@5", ["image_to_thermal_R@5", "image_to_thermal_r5", "i2t_R@5"], "Visible image-to-thermal Recall@5"),
        ("image_to_thermal_R@10", ["image_to_thermal_R@10", "image_to_thermal_r10", "i2t_R@10"], "Visible image-to-thermal Recall@10"),
        ("thermal_to_image_R@1", ["thermal_to_image_R@1", "thermal_to_image_r1", "t2i_R@1"], "Thermal-to-visible image Recall@1"),
        ("thermal_to_image_R@5", ["thermal_to_image_R@5", "thermal_to_image_r5", "t2i_R@5"], "Thermal-to-visible image Recall@5"),
        ("thermal_to_image_R@10", ["thermal_to_image_R@10", "thermal_to_image_r10", "t2i_R@10"], "Thermal-to-visible image Recall@10"),
        ("accuracy", ["accuracy", "test_accuracy", "top1", "top1_acc"], "Optional classification/top-1 accuracy"),
        ("top1", ["top1", "top1_acc", "test_top1"], "Optional top-1 accuracy"),
    ],
    "nyu": [
        ("image_to_depth_R@1", ["image_to_depth_R@1", "image_to_depth_r1", "i2d_r@1", "I2D R@1"], "RGB image-to-depth Recall@1"),
        ("image_to_depth_R@5", ["image_to_depth_R@5", "image_to_depth_r5", "i2d_r@5", "I2D R@5"], "RGB image-to-depth Recall@5"),
        ("image_to_depth_R@10", ["image_to_depth_R@10", "image_to_depth_r10", "i2d_r@10", "I2D R@10"], "RGB image-to-depth Recall@10"),
        ("depth_to_image_R@1", ["depth_to_image_R@1", "depth_to_image_r1", "d2i_r@1", "D2I R@1"], "Depth-to-RGB image Recall@1"),
        ("depth_to_image_R@5", ["depth_to_image_R@5", "depth_to_image_r5", "d2i_r@5", "D2I R@5"], "Depth-to-RGB image Recall@5"),
        ("depth_to_image_R@10", ["depth_to_image_R@10", "depth_to_image_r10", "d2i_r@10", "D2I R@10"], "Depth-to-RGB image Recall@10"),
        ("top1", ["top1", "top1_acc", "test_top1"], "Optional top-1 classification accuracy"),
        ("top5", ["top5", "top5_acc", "test_top5"], "Optional top-5 classification accuracy"),
    ],
    "uci": [
        ("accuracy", ["accuracy", "test_accuracy", "val_accuracy"], "Human activity recognition accuracy"),
        ("macro_f1", ["macro_f1", "test_macro_f1", "f1_macro"], "Macro-averaged F1 score"),
        ("per_class_accuracy", ["per_class_accuracy", "class_accuracy", "per_class_acc"], "Per-class accuracy breakdown if available"),
    ],
}

HEADER_FILL = PatternFill("solid", fgColor="1F4E78")
TITLE_FILL = PatternFill("solid", fgColor="D9EAF7")
ALT_FILL = PatternFill("solid", fgColor="F7FBFF")
WHITE_FONT = Font(name="Arial", color="FFFFFF", bold=True)
TITLE_FONT = Font(name="Arial", size=14, bold=True, color="1F4E78")
BASE_FONT = Font(name="Arial", size=10)
THIN_SIDE = Side(style="thin", color="B7B7B7")
THIN_BORDER = Border(left=THIN_SIDE, right=THIN_SIDE, top=THIN_SIDE, bottom=THIN_SIDE)
STATUS_FILLS = {
    "GOOD": PatternFill("solid", fgColor="C6EFCE"),
    "PRESENT": PatternFill("solid", fgColor="C6EFCE"),
    "REVIEW": PatternFill("solid", fgColor="FCE4D6"),
    "LOW": PatternFill("solid", fgColor="FFC7CE"),
    "NOT_AVAILABLE": PatternFill("solid", fgColor="E7E6E6"),
    "WARNING": PatternFill("solid", fgColor="FCE4D6"),
}

def normalize_key(value):
    return "".join(ch.lower() for ch in str(value) if ch.isalnum())

def load_json_file(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f), None
    except Exception as exc:
        return None, f"WARNING: failed to read {path.name}: {exc}"

def ordered_json_files():
    priority = [
        "final_eval_summary.json",
        "eval_results.json",
        "retrieval_recall.json",
        "train_history.json",
        "ablation_summary.json",
    ]
    files = []
    seen = set()
    for name in priority:
        p = EXCEL_OUTPUT_DIR / name
        if p.exists() and p.is_file():
            files.append(p)
            seen.add(p.resolve())
    for p in sorted(EXCEL_OUTPUT_DIR.rglob("*.json")):
        try:
            key = p.resolve()
        except Exception:
            key = p
        if key not in seen:
            files.append(p)
            seen.add(key)
    return files

JSON_OBJECTS = []
JSON_WARNINGS = []
for json_path in ordered_json_files():
    obj, warning = load_json_file(json_path)
    if warning:
        JSON_WARNINGS.append(warning)
    else:
        JSON_OBJECTS.append((json_path.name, obj))

TXT_WARNINGS = []
for txt_path in sorted(list(EXCEL_OUTPUT_DIR.rglob("*.txt")) + list(EXCEL_OUTPUT_DIR.rglob("*.md"))):
    try:
        text = txt_path.read_text(encoding="utf-8", errors="replace")
        if "WARNING" in text.upper() or "NOT_AVAILABLE" in text.upper() or "NOT_RUN" in text.upper():
            TXT_WARNINGS.append(f"See {txt_path.name} for warnings/status notes")
    except Exception as exc:
        TXT_WARNINGS.append(f"WARNING: failed to read {txt_path.name}: {exc}")

def flatten_json(obj, prefix="", source=""):
    rows = []
    if prefix:
        rows.append((prefix, obj, source))
    if isinstance(obj, dict):
        for key, value in obj.items():
            child = f"{prefix}.{key}" if prefix else str(key)
            rows.extend(flatten_json(value, child, source))
    elif isinstance(obj, list):
        for idx, value in enumerate(obj):
            child = f"{prefix}.{idx}" if prefix else str(idx)
            rows.extend(flatten_json(value, child, source))
    return rows

FLAT_JSON = []
for source, obj in JSON_OBJECTS:
    FLAT_JSON.extend(flatten_json(obj, "", source))

def find_value(candidates):
    normalized_candidates = [normalize_key(c) for c in candidates]
    for cand in normalized_candidates:
        for key, value, source in FLAT_JSON:
            last_part = key.split(".")[-1]
            if normalize_key(last_part) == cand:
                return value, f"source: {source} / {key}"
    for cand in normalized_candidates:
        if len(cand) < 6:
            continue
        for key, value, source in FLAT_JSON:
            if normalize_key(key).endswith(cand):
                return value, f"source: {source} / {key}"
    return None, "NOT_AVAILABLE"

def safe_float(value):
    if isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        try:
            return float(value)
        except Exception:
            return None
    return None

def display_value(value):
    if value is None:
        return "NOT_AVAILABLE"
    if isinstance(value, bool):
        return str(value)
    if isinstance(value, (int, float)):
        return round(float(value), 6)
    if isinstance(value, (dict, list)):
        rendered = json.dumps(value, ensure_ascii=False)
        return rendered[:300] + ("..." if len(rendered) > 300 else "")
    return str(value)

def status_for(metric, value):
    if value is None or value == "NOT_AVAILABLE":
        return "NOT_AVAILABLE"
    if isinstance(value, (dict, list)):
        return "PRESENT" if value else "NOT_AVAILABLE"
    score = safe_float(value)
    if score is None:
        return "PRESENT"
    if score > 1.0 and score <= 100.0:
        score = score / 100.0
    metric_l = str(metric).lower()
    if "loss" in metric_l:
        if score <= 0.5:
            return "GOOD"
        if score <= 1.5:
            return "REVIEW"
        return "LOW"
    if score >= 0.70:
        return "GOOD"
    if score >= 0.40:
        return "REVIEW"
    return "LOW"

def get_by_alias(row, aliases):
    if not isinstance(row, dict):
        return None
    lookup = {normalize_key(k): v for k, v in row.items()}
    for alias in aliases:
        key = normalize_key(alias)
        if key in lookup:
            return lookup[key]
    return None

def find_train_history():
    candidates = [EXCEL_OUTPUT_DIR / "train_history.json"]
    candidates.extend(sorted(EXCEL_OUTPUT_DIR.rglob("*train*history*.json")))
    seen = set()
    for path in candidates:
        try:
            key = path.resolve()
        except Exception:
            key = path
        if key in seen or not path.exists():
            continue
        seen.add(key)
        obj, warning = load_json_file(path)
        if warning:
            JSON_WARNINGS.append(warning)
            continue
        if isinstance(obj, list):
            return obj, path.name
        if isinstance(obj, dict):
            for key in ["history", "train_history", "rows"]:
                if isinstance(obj.get(key), list):
                    return obj[key], path.name
    return None, None

def build_training_rows():
    history, source = find_train_history()
    if not history:
        return ["Status", "Notes"], [["NOT_AVAILABLE", "train_history.json not found in output folder"]]
    alias_map = {
        "epoch": ["epoch"],
        "train_loss": ["train_loss", "loss_train", "training_loss"],
        "val_loss": ["val_loss", "test_loss", "valid_loss", "validation_loss"],
        "train_top1": ["train_top1", "train_accuracy", "train_acc", "train_i2t_acc", "train_top1_acc"],
        "val_top1": ["val_top1", "val_accuracy", "val_acc", "test_accuracy", "val_i2t_acc", "test_top1"],
        "lr": ["lr", "learning_rate"],
    }
    normalized_rows = []
    for raw in history:
        if not isinstance(raw, dict):
            continue
        row = {col: get_by_alias(raw, aliases) for col, aliases in alias_map.items()}
        normalized_rows.append(row)
    columns = [col for col in ["epoch", "train_loss", "val_loss", "train_top1", "val_top1", "lr"] if any(row.get(col) is not None for row in normalized_rows)]
    if "epoch" not in columns:
        columns = ["epoch"] + columns
    rows = [[display_value(row.get(col)) for col in columns] for row in normalized_rows]
    best_row = None
    best_note = None
    val_top1_values = [(safe_float(row.get("val_top1")), idx) for idx, row in enumerate(normalized_rows)]
    val_top1_values = [(v, idx) for v, idx in val_top1_values if v is not None]
    if val_top1_values:
        _, idx = max(val_top1_values, key=lambda x: x[0])
        best_row = normalized_rows[idx]
        best_note = "Best Epoch by val_top1"
    else:
        val_loss_values = [(safe_float(row.get("val_loss")), idx) for idx, row in enumerate(normalized_rows)]
        val_loss_values = [(v, idx) for v, idx in val_loss_values if v is not None]
        if val_loss_values:
            _, idx = min(val_loss_values, key=lambda x: x[0])
            best_row = normalized_rows[idx]
            best_note = "Best Epoch by val_loss"
    if best_row:
        best = [display_value(best_row.get(col)) for col in columns]
        if best:
            best[0] = f"Best Epoch: {best[0]}"
        rows.append(best)
        rows.append([best_note] + [""] * (len(columns) - 1))
    rows.append([f"source: {source}"] + [""] * (len(columns) - 1))
    return columns, rows

def find_ablation_object():
    candidates = [EXCEL_OUTPUT_DIR / "ablation_summary.json"]
    candidates.extend(sorted(EXCEL_OUTPUT_DIR.rglob("*ablation*.json")))
    seen = set()
    for path in candidates:
        try:
            key = path.resolve()
        except Exception:
            key = path
        if key in seen or not path.exists():
            continue
        seen.add(key)
        obj, warning = load_json_file(path)
        if warning:
            JSON_WARNINGS.append(warning)
            continue
        return obj, path.name
    return None, None

def setting_name(record, index):
    if not isinstance(record, dict):
        return f"setting_{index}"
    for key in ["setting", "name", "temperature", "lr", "learning_rate", "hidden_dim", "batch_size"]:
        if key in record:
            return f"{key}={record[key]}" if key not in ["setting", "name"] else str(record[key])
    return f"setting_{index}"

def build_ablation_rows():
    obj, source = find_ablation_object()
    if obj is None:
        return [["NOT_AVAILABLE", "NOT_AVAILABLE", "NOT_AVAILABLE", "No ablation JSON found in output folder"]]
    records = obj if isinstance(obj, list) else obj.get("results", obj.get("ablation", [obj])) if isinstance(obj, dict) else []
    if isinstance(records, dict):
        records = [records]
    rows = []
    for idx, record in enumerate(records):
        if not isinstance(record, dict):
            rows.append([f"setting_{idx}", "value", display_value(record), f"source: {source}"])
            continue
        flat = flatten_json(record, "", source)
        numeric_items = []
        for key, value, _ in flat:
            if safe_float(value) is not None and key.split(".")[-1].lower() not in ["epoch", "epochs", "seed"]:
                numeric_items.append((key, value))
        if numeric_items:
            for key, value in numeric_items[:12]:
                rows.append([setting_name(record, idx), key, display_value(value), f"source: {source}"])
        else:
            note = record.get("reason", record.get("note", "No numeric metric found"))
            status = record.get("status", "NOT_AVAILABLE")
            rows.append([setting_name(record, idx), "status", status, f"{note}; source: {source}"])
    return rows or [["NOT_AVAILABLE", "NOT_AVAILABLE", "NOT_AVAILABLE", f"No ablation records in {source}"]]

def find_per_class():
    for key, value, source in FLAT_JSON:
        if normalize_key(key.split(".")[-1]) in ["perclassaccuracy", "classaccuracy", "perclassacc"]:
            return value, source, key
    return None, None, None

def build_per_class_rows():
    value, source, key = find_per_class()
    if value is None:
        return [["NOT_AVAILABLE", "NOT_AVAILABLE", "NOT_AVAILABLE"]]
    rows = []
    if isinstance(value, dict):
        for cls, acc in value.items():
            rows.append([str(cls), display_value(acc), status_for("accuracy", acc)])
    elif isinstance(value, list):
        for idx, acc in enumerate(value):
            rows.append([str(idx), display_value(acc), status_for("accuracy", acc)])
    else:
        rows.append(["per_class_accuracy", display_value(value), status_for("accuracy", value)])
    rows.append([f"source: {source}", key, ""])
    return rows

def purpose_for(path):
    suffix = path.suffix.lower()
    if suffix == ".json":
        return "metrics/data"
    if suffix == ".png":
        return "figure/chart"
    if suffix in [".txt", ".md"]:
        return "report"
    if suffix == ".xlsx":
        return "excel report"
    if suffix == ".zip":
        return "compressed output"
    if suffix in [".pth", ".pt", ".ckpt"]:
        return "checkpoint"
    return "output artifact"

def build_manifest_rows():
    rows = []
    for path in sorted(EXCEL_OUTPUT_DIR.rglob("*")):
        if not path.is_file():
            continue
        try:
            rel = str(path.relative_to(EXCEL_OUTPUT_DIR))
        except Exception:
            rel = path.name
        try:
            size_kb = round(path.stat().st_size / 1024, 2)
        except Exception:
            size_kb = "NOT_AVAILABLE"
        rows.append([rel, path.suffix.lower() or "NO_EXTENSION", size_kb, purpose_for(path)])
    return rows or [["NOT_AVAILABLE", "NOT_AVAILABLE", "NOT_AVAILABLE", "Output folder is empty"]]

def autosize(ws):
    for col_idx, column_cells in enumerate(ws.columns, start=1):
        max_len = 0
        for cell in column_cells:
            try:
                max_len = max(max_len, len(str(cell.value)) if cell.value is not None else 0)
            except Exception:
                pass
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len + 2, 12), 55)

def create_table_sheet(wb, title, headers, rows):
    ws = wb.create_sheet(title)
    ws.append([title] + [""] * (len(headers) - 1))
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(headers))
    title_cell = ws.cell(row=1, column=1)
    title_cell.font = TITLE_FONT
    title_cell.fill = TITLE_FILL
    title_cell.alignment = Alignment(horizontal="center")
    ws.append([""] * len(headers))
    ws.append(headers)
    for row in rows:
        fixed = list(row)[:len(headers)] + [""] * max(0, len(headers) - len(row))
        ws.append(fixed)
    for row in ws.iter_rows():
        for cell in row:
            cell.font = BASE_FONT
            cell.border = THIN_BORDER
            cell.alignment = Alignment(vertical="top", wrap_text=True)
    title_cell.font = TITLE_FONT
    title_cell.fill = TITLE_FILL
    for cell in ws[3]:
        cell.fill = HEADER_FILL
        cell.font = WHITE_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    status_cols = [idx + 1 for idx, name in enumerate(headers) if "status" in name.lower()]
    for row_idx in range(4, ws.max_row + 1):
        if row_idx % 2 == 0:
            for col_idx in range(1, len(headers) + 1):
                ws.cell(row=row_idx, column=col_idx).fill = ALT_FILL
        for col_idx in status_cols:
            value = str(ws.cell(row=row_idx, column=col_idx).value or "")
            fill = STATUS_FILLS.get(value)
            if fill:
                ws.cell(row=row_idx, column=col_idx).fill = fill
    ws.freeze_panes = "A4"
    autosize(ws)
    return ws

metric_rows = []
for metric, candidates, meaning in METRIC_SPECS[METRIC_SET]:
    value, note = find_value([metric] + candidates)
    metric_rows.append([metric, display_value(value), meaning, status_for(metric, value), note])
if JSON_WARNINGS or TXT_WARNINGS:
    for warning in JSON_WARNINGS[:5] + TXT_WARNINGS[:5]:
        metric_rows.append(["warning", warning, "Runtime/file warning", "WARNING", "Read from JSON/TXT files only"])

training_headers, training_rows = build_training_rows()
ablation_rows = build_ablation_rows()
per_class_rows = build_per_class_rows()

wb = Workbook()
wb.remove(wb.active)
create_table_sheet(wb, "Evaluation Metrics", ["Metric", "Value", "Benchmark/Meaning", "Status", "Notes"], metric_rows)
create_table_sheet(wb, "Training Summary", training_headers, training_rows)
create_table_sheet(wb, "Ablation Summary", ["Setting", "Metric", "Value", "Notes"], ablation_rows)
per_class_ws = create_table_sheet(wb, "Per-Class Breakdown", ["Class", "Accuracy", "Status"], per_class_rows)
per_class_ws.cell(row=1, column=1).value = "Per-Class / Breakdown"

# Save once so the manifest can include the Excel report itself.
wb.save(EXCEL_REPORT_PATH)
manifest_rows = build_manifest_rows()
create_table_sheet(wb, "Output Manifest", ["File name", "Extension", "Size KB", "Purpose"], manifest_rows)
wb.save(EXCEL_REPORT_PATH)

print(f"Excel report saved to: {EXCEL_REPORT_PATH}")
print(f"Total sheets: {len(wb.sheetnames)}")
print(f"Total output files listed: {len(manifest_rows)}")

## ADDON: Zip this notebook output

This cell zips this notebook's standardized output folder for direct Kaggle download. It skips missing folders safely and excludes large checkpoints over 100MB.

In [ ]:
import os
import zipfile
from pathlib import Path

def zip_folder(output_dir, zip_path, max_checkpoint_mb=100):
    output_dir = Path(output_dir)
    zip_path = Path(zip_path)

    if not output_dir.exists():
        print(f"WARNING: output folder does not exist: {output_dir}")
        return None

    added_files = []
    skipped_files = []

    def should_skip(file_path):
        suffix = file_path.suffix.lower()
        if suffix in [".pth", ".pt", ".ckpt"]:
            size_mb = file_path.stat().st_size / (1024 * 1024)
            if size_mb > max_checkpoint_mb:
                return True
        return False

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for file_path in output_dir.rglob("*"):
            if not file_path.is_file():
                continue

            if should_skip(file_path):
                skipped_files.append(str(file_path))
                continue

            arcname = file_path.relative_to(output_dir.parent)
            zipf.write(file_path, arcname)
            added_files.append(str(arcname))

    size_mb = zip_path.stat().st_size / (1024 * 1024)

    print("=" * 80)
    print(f"ZIP CREATED: {zip_path}")
    print(f"ZIP SIZE: {size_mb:.2f} MB")
    print(f"TOTAL FILES ADDED: {len(added_files)}")
    print("=" * 80)

    for f in added_files:
        print(f"- {f}")

    if skipped_files:
        print("=" * 80)
        print("SKIPPED LARGE CHECKPOINTS:")
        for f in skipped_files:
            print(f"- {f}")

    print("=" * 80)
    print("Download path:")
    print(str(zip_path))

    return zip_path

zip_folder(
    "/kaggle/working/ResultForDepth4",
    "/kaggle/working/ResultForDepth4.zip"
)